In [1]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import train_test_split 
from sklearn.metrics import mean_squared_error
import pandas as pd 
import numpy as np
import mlflow

c:\Users\HP\Documents\projects\Prediction de l’Esperance de Vie– Modelisation Machine Learning & EDA & MLOPs\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
df_train_x = pd.read_csv("./../data/processed/train_df_x.csv")
df_train_y = pd.read_csv("./../data/processed/train_df_y.csv")


df_train_x , df_val_x , df_train_y , df_val_y = train_test_split(df_train_x , df_train_y , random_state=123 , test_size=0.15)

In [3]:
# Doit afficher le chemin du package dans site-packages, PAS votre dossier local
print(mlflow.__file__)

c:\Users\HP\Documents\projects\Prediction de l’Esperance de Vie– Modelisation Machine Learning & EDA & MLOPs\.venv\Lib\site-packages\mlflow\__init__.py


In [ ]:
mlflow.set_experiment(experiment_name="trainign_model_Esperance_de_vie")

with mlflow.start_run():
    exp_l1_ratio = 0.5
    exp_alpha = 1
    exp_num_ite = 1000
    model_lr = ElasticNet(alpha=exp_alpha , l1_ratio=exp_l1_ratio , fit_intercept=True , max_iter=exp_num_ite,random_state=123)
    model_lr.fit(df_train_x , df_train_y)

    train_predicted_y = model_lr.predict(df_train_x)
    val_predicted_y = model_lr.predict(df_val_x)

    R_train_mse = np.sqrt(mean_squared_error(df_train_y , train_predicted_y))
    R_val_mse = np.sqrt(mean_squared_error(df_val_y , val_predicted_y) )

    mlflow.log_param("exp_l1_ratio" , exp_l1_ratio)
    mlflow.log_param("exp_alpha" , exp_alpha)
    mlflow.log_param("exp_num_ite" , exp_num_ite)
    mlflow.sklearn.log_model(model_lr , "model_ElasticNet")
    mlflow.log_metric("rmse/num_obs" , R_val_mse/len(train_predicted_y))

    print(f"---- train mse : {R_train_mse/len(train_predicted_y)}")
    print(f"---- val mse : {R_val_mse/len(train_predicted_y)}")

    

2026/09/27 18:02:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/27 18:03:02 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


---- train mse : 190.67297559402473
---- val mse : 192.64788531193804


In [5]:
# mlflow.set_experiment(experiment_name="trainign_model_Esperance_de_vie")

# with mlflow.start_run():
# Random Forest Model Training 
n_estimators = 100
best_rf_model = None
best_max_depth = 15 # c'est ce qu'on doit minimiser si on a un sur-apprentissage et maximiser si on une degradation (le model n'a pas reussit a capturer les patterns)
best_min_samples_split = 2 # to increase if we have a overfitting
best_min_samples_leaf = 1 # to increase if our model can't learn
criterion = "squared_error"
best_mae = 100000

for epoc in range(15):
    rf_model = RandomForestRegressor(n_estimators=n_estimators , criterion=criterion , max_depth=best_max_depth, min_samples_leaf=best_min_samples_leaf , min_samples_split = best_min_samples_split )
    # 2. Modèle sans restriction (très susceptible de surapprendre)
    scoring = {
    'r2': 'r2',
    'mae': 'neg_mean_absolute_error'
    }
    # 3. Validation croisée en demandant le score d'entraînement
    cv_results = cross_validate(
    rf_model, 
    df_train_x, 
    df_train_y, 
    cv=5, 
    scoring=scoring, 
    return_train_score=True
    )
    print(cv_results)
    # 4. Calcul des performances moyennes
    train_r2 = np.mean(cv_results['train_r2'])
    val_r2 = np.mean(cv_results['test_r2'])

    train_mae = -np.mean(cv_results['train_mae'])
    val_mae = -np.mean(cv_results['test_mae'])

    print("---- End Training ")
    if train_mae < val_mae and best_mae > val_mae:
        print("A Better Model Was found...")
        best_mae = val_mae
        best_rf_model = rf_model
        if best_mae < 10:
            print("The best Model found... End-----")
            break

    if train_mae >= val_mae :
        print("There is a OverFitting...")
        best_max_depth = best_max_depth -2
        best_min_samples_split = best_min_samples_split +2
        continue

    
    print(f"-----------epoch : {epoc} ----")
    print(f"--- R² Score (Proche de 1 = Meilleur) ---")
    print(f"Train R² : {train_r2:.4f}")
    print(f"Val R²   : {val_r2:.4f}")
    print(f"Écart R² : {train_r2 - val_r2:.4f}\n")

    print(f"--- MAE / Erreur Moyenne (Proche de 0 = Meilleur) ---")
    print(f"Train MAE : {train_mae:.2f}")
    print(f"Val MAE   : {val_mae:.2f}")


NameError: name 'cross_validate' is not defined